In [2]:
import logging
import warnings
from datetime import datetime
import os
import sys
import numpy as np
import pandas as pd
import matplotlib as mlp
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

In [3]:
today = datetime.now().date()

In [4]:
logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s - %(levelname)s - %(message)s", 
    datefmt="%Y-%m-%d %H:%M:%S", 
    handlers=[
        logging.FileHandler(f"./logs/msc_data_generation_{today}.log", mode='w'), 
        logging.StreamHandler(sys.stdout)
    ]
)

Each of the subjects were shown a sequence of indoor and outdoor images each session. Now, normally, the order of this sequence is randomized. We need to hopefully find a few that are the same sequence of stimuli. We will know that those specific sessions should be the same. Thus, if $sub^{n}_{i}$, $sub^{n+x}_{j}$, and $sub^{n+y}_{k}$, with superscript referring to subject number and subscript referring to session number, all have the same sequence in their event files, we can assume that all differences between them can be discarded. We can also take the similarities between the discarded information as the inverse of the transformation all other sessions must go through to be in the same space.

In [5]:
# ---Variable Config---
config = {
    'data_dir': "../data/fmriprep_output/derivatives",
    'raw_dir': "../data/ds000224",
    'roi_mask_path': "./Sphere_Mask.nii",
    'subj_prefix': "sub-MSC",
    'sess_prefix': "ses-func",
    'task_label': "task-memoryscenes",
    'space_label': "space-MNI152NLin2009cAsym",
    'bold_suffix': "desc-preproc_bold.nii.gz",
    'mask_suffix': "desc-brain_mask.nii.gz",
    'confounds_suffix': "desc-confounds_timeseries.tsv",
    'events_suffix': 'events.tsv',
    'data_output_dir': 'datasets', 
    'roi_label': 'roi-fef_r',
    'encoding_label': 'encoding-continuous',
    'csv_suffix': 'desc-vox_w_target.csv',
    'nilearn_defaults': {
         'standardize': 'zscore_sample',
         'detrend': True,
         'low_pass': 0.1,
         'high_pass': 0.01,
         't_r': 2
        },
    'hrf_model': 'spm'
}

In [9]:
df_list = []
for sub in range(1,11):
    if sub==10:
        sub = str(sub)
    else:
        sub = "0"+str(sub)
    
    for ses in range(1,15):
        if ses>9:
            ses = str(ses)
        else:
            ses = "0"+str(ses)

        p_events = os.path.join(config['raw_dir'], 
                        config['subj_prefix']+sub, 
                        config['sess_prefix']+ses, 
                        'func',  
                        "_".join([config['subj_prefix']+sub, 
                                  config['sess_prefix']+ses, 
                                  config['task_label'], 
                                  config['events_suffix']
                                 ]
                                )
                       )

        if not os.path.isfile(p_events):
            print(f"Failed: {p_events}")
            continue

        df = pd.read_csv(p_events, header=0, sep="\t")
        df['subj'] = [sub]*df.shape[0]
        df['sess'] = [ses]*df.shape[0]
        df_list.append(df)
        print(f"Subject {sub}, session {ses}, accumulated dfs {len(df_list)}")

edf = pd.concat(df_list, axis=0, ignore_index=True)

Subject 01, session 01, accumulated dfs 1
Subject 01, session 02, accumulated dfs 2
Subject 01, session 03, accumulated dfs 3
Subject 01, session 04, accumulated dfs 4
Subject 01, session 05, accumulated dfs 5
Subject 01, session 06, accumulated dfs 6
Subject 01, session 07, accumulated dfs 7
Subject 01, session 08, accumulated dfs 8
Subject 01, session 09, accumulated dfs 9
Subject 01, session 10, accumulated dfs 10
Failed: ../data/ds000224/sub-MSC01/ses-func11/func/sub-MSC01_ses-func11_task-memoryscenes_events.tsv
Failed: ../data/ds000224/sub-MSC01/ses-func12/func/sub-MSC01_ses-func12_task-memoryscenes_events.tsv
Failed: ../data/ds000224/sub-MSC01/ses-func13/func/sub-MSC01_ses-func13_task-memoryscenes_events.tsv
Failed: ../data/ds000224/sub-MSC01/ses-func14/func/sub-MSC01_ses-func14_task-memoryscenes_events.tsv
Subject 02, session 01, accumulated dfs 11
Subject 02, session 02, accumulated dfs 12
Subject 02, session 03, accumulated dfs 13
Failed: ../data/ds000224/sub-MSC02/ses-func04/

In [8]:
edf.trial_type.unique()

array(['indoor_corr_1', 'outdoor_corr_1', 'err', 'indoor_corr_2',
       'outdoor_corr_2', 'indoor_corr_3', 'outdoor_corr_3', 'unknown'],
      dtype=object)

In [10]:
from sklearn.preprocessing import OneHotEncoder
encode = OneHotEncoder()

In [14]:
tmp = encode.fit_transform(edf)

In [17]:
tmp.shape

(6984, 1194)

In [18]:
edf.head()

,onset,duration,trial_type,response_time,correct,subj,sess
0,11.0,2.2,indoor_corr_1,1.126,corr,01,01
1,13.2,2.2,indoor_corr_1,1.199,corr,01,01
2,15.4,2.2,indoor_corr_1,1.031,corr,01,01
3,19.8,2.2,outdoor_corr_1,1.073,corr,01,01
4,22.0,2.2,outdoor_corr_1,0.797,corr,01,01


---